
# Task 3 - Modular Logged ETL Pipeline

This notebook implements a production-style ETL pipeline using the TVMaze API.

### Features Included
- Modular ETL functions:
  - `extract()`
  - `clean()`
  - `transform()`
  - `load()`
- Full logging support
- Error handling
- Data cleaning
- Feature engineering
- Summary table using `groupby()`
- CSV export
- MySQL loading with idempotency


In [2]:

import requests
import pandas as pd
import logging
import mysql.connector
from mysql.connector import Error
import os

# ---------------- LOGGING CONFIG ----------------
logging.basicConfig(
    filename='etl_pipeline.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger()

API_URL = "https://api.tvmaze.com/shows"
CSV_FILE = "tvmaze_cleaned.csv"

# ---------------- MYSQL CONFIG ----------------
MYSQL_HOST = "localhost"
MYSQL_USER = "root"
MYSQL_PASSWORD = os.getenv("DB_PASSWORD")
MYSQL_DATABASE = "etl_project"
MYSQL_TABLE = "tvmaze_shows"


## 1. Extract Function

In [3]:

def extract(url, timeout=10):
    logger.info("Starting extraction process")

    try:
        response = requests.get(url, timeout=timeout)
        response.raise_for_status()

        data = response.json()

        df = pd.DataFrame(data)

        logger.info(f"Extraction completed successfully. Rows extracted: {len(df)}")

        return df

    except requests.exceptions.Timeout:
        logger.error("Request timed out")

    except requests.exceptions.HTTPError as e:
        logger.error(f"HTTP Error: {e}")

    except requests.exceptions.ConnectionError:
        logger.error("Connection Error")

    except Exception as e:
        logger.error(f"Unexpected Error: {e}")

    return pd.DataFrame()


## 2. Clean Function

In [4]:

def clean(df):
    logger.info(f"Cleaning started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in clean()")
        return pd.DataFrame()

    cleaned_df = df.copy()

    # ---------------- NORMALIZE NESTED COLUMNS ----------------
    cleaned_df['rating_average'] = cleaned_df['rating'].apply(
        lambda x: x.get('average') if isinstance(x, dict) else None
    )

    cleaned_df['schedule_time'] = cleaned_df['schedule'].apply(
        lambda x: x.get('time') if isinstance(x, dict) else None
    )

    cleaned_df['schedule_days'] = cleaned_df['schedule'].apply(
        lambda x: ', '.join(x.get('days')) if isinstance(x, dict) else None
    )

    # ---------------- REMOVE UNUSED COLUMNS ----------------
    drop_columns = [
        'runtime',
        'averageRuntime',
        'officialSite',
        'weight',
        'webChannel',
        'updated',
        'rating',
        'schedule',
        'network',
        'image',
        '_links',
        'externals',
        'dvdCountry'
    ]

    existing_cols = [col for col in drop_columns if col in cleaned_df.columns]

    cleaned_df.drop(columns=existing_cols, inplace=True)

    # ---------------- HANDLE NULL VALUES ----------------
    cleaned_df['rating_average'] = cleaned_df['rating_average'].fillna(0)

    object_columns = cleaned_df.select_dtypes(include='object').columns

    for col in object_columns:
        cleaned_df[col] = cleaned_df[col].fillna('Unknown')

    # ---------------- DATE CONVERSION ----------------
    for col in ['premiered', 'ended']:
        if col in cleaned_df.columns:
            cleaned_df[col] = pd.to_datetime(cleaned_df[col], errors='coerce')

    # ---------------- REMOVE HTML TAGS ----------------
    if 'summary' in cleaned_df.columns:
        cleaned_df['summary'] = cleaned_df['summary'].str.replace(
            r'<.*?>', '',
            regex=True
        )

    logger.info(f"Cleaning completed. Output rows: {len(cleaned_df)}")

    return cleaned_df


## 3. Transform Function

In [5]:

def transform(df):
    logger.info(f"Transformation started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in transform()")
        return pd.DataFrame(), pd.DataFrame()

    transformed_df = df.copy()

    # ---------------- FEATURE ENGINEERING ----------------

    # 1. Premier Year
    transformed_df['premier_year'] = transformed_df['premiered'].dt.year.fillna(0)

    # 2. Rating Category
    transformed_df['rating_category'] = transformed_df['rating_average'].apply(
        lambda x: (
            'Excellent' if x >= 8 else
            'Good' if x >= 6 else
            'Average' if x >= 4 else
            'Poor'
        )
    )

    # 3. Summary Word Count
    transformed_df['summary_word_count'] = transformed_df['summary'].apply(
        lambda x: len(str(x).split())
    )

    # 4. Show Duration
    transformed_df['show_duration_years'] = (
        transformed_df['ended'].dt.year.fillna(
            pd.Timestamp.now().year
        ) - transformed_df['premiered'].dt.year.fillna(0)
    )

    logger.info("Feature engineering completed")

    # ---------------- SUMMARY TABLE ----------------
    summary_table = transformed_df.groupby('language').agg(
        average_rating=('rating_average', 'mean'),
        min_rating=('rating_average', 'min'),
        max_rating=('rating_average', 'max'),
        total_shows=('id', 'count')
    ).reset_index()

    logger.info("Summary table generated")

    logger.info(f"Transformation completed. Output rows: {len(transformed_df)}")

    return transformed_df, summary_table


## 4. Load Function

In [6]:

def load(df, csv_file, table_name):
    logger.info(f"Loading started. Input rows: {len(df)}")

    if df.empty:
        logger.warning("Empty DataFrame received in load()")
        return

    # ---------------- SAVE CSV ----------------
    try:
        df.to_csv(csv_file, index=False)

        logger.info(f"CSV saved successfully: {csv_file}")

    except Exception as e:
        logger.error(f"CSV Save Error: {e}")

    # ---------------- MYSQL LOAD ----------------
    try:
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            user=MYSQL_USER,
            password=MYSQL_PASSWORD
        )

        cursor = conn.cursor()

        cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}")
        cursor.execute(f"USE {MYSQL_DATABASE}")

        # Create table
        create_table_query = f'''
        CREATE TABLE IF NOT EXISTS {table_name} (
            id INT PRIMARY KEY,
            name VARCHAR(255),
            language VARCHAR(100),
            status VARCHAR(100),
            genres TEXT,
            premiered DATETIME,
            ended DATETIME,
            rating_average FLOAT,
            rating_category VARCHAR(50),
            summary_word_count INT,
            show_duration_years INT
        )
        '''

        cursor.execute(create_table_query)

        rows_inserted = 0

        insert_query = f'''
        INSERT IGNORE INTO {table_name}
        (
            id,
            name,
            language,
            status,
            genres,
            premiered,
            ended,
            rating_average,
            rating_category,
            summary_word_count,
            show_duration_years
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
        '''

        for _, row in df.iterrows():
            values = (
                int(row['id']),
                str(row.get('name', '')),
                str(row.get('language', '')),
                str(row.get('status', '')),
                str(row.get('genres', '')),
                row.get('premiered'),
                row.get('ended'),
                float(row.get('rating_average', 0)),
                str(row.get('rating_category', '')),
                int(row.get('summary_word_count', 0)),
                int(row.get('show_duration_years', 0))
            )

            cursor.execute(insert_query, values)

            rows_inserted += cursor.rowcount

        conn.commit()

        logger.info(f"MySQL loading completed. Rows inserted: {rows_inserted}")

    except Error as e:
        logger.error(f"MySQL Error: {e}")

    finally:
        try:
            cursor.close()
            conn.close()
            logger.info("MySQL connection closed")
        except:
            pass


## 5. Run ETL Pipeline

In [7]:

def run_pipeline():
    logger.info("ETL Pipeline Started")

    # Extract
    raw_df = extract(API_URL)

    if raw_df.empty:
        logger.error("Pipeline stopped during extraction")
        return

    # Clean
    cleaned_df = clean(raw_df)

    if cleaned_df.empty:
        logger.error("Pipeline stopped during cleaning")
        return

    # Transform
    transformed_df, summary_df = transform(cleaned_df)

    if transformed_df.empty:
        logger.error("Pipeline stopped during transformation")
        return

    print("Summary Table")
    print(summary_df.head())

    # Load
    load(transformed_df, CSV_FILE, MYSQL_TABLE)

    logger.info("ETL Pipeline Completed")

    print("ETL Pipeline Completed Successfully")
    print("Log file created: etl_pipeline.log")


In [8]:

# Execute pipeline
run_pipeline()


Summary Table
   language  average_rating  min_rating  max_rating  total_shows
0   English        7.452966         0.0         9.2          236
1  Japanese        7.850000         7.3         8.8            4
ETL Pipeline Completed Successfully
Log file created: etl_pipeline.log


C:\Users\sumit\AppData\Local\Temp\ipykernel_17316\4070028895.py:47: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_columns = cleaned_df.select_dtypes(include='object').columns
